<a href="https://colab.research.google.com/github/2024akcs4377f-spec/ML-assignment-2/blob/cross-validation/cross_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Machine Learning Workflow: Cross-Validation, Tuning, Pipelines, and Comparison

This notebook provides a scaffold to perform cross-validation, tune hyperparameters, build pipelines, and compare different machine learning models. Please replace the placeholder sections with your specific data loading and model definitions from your `ridge_and_SVM.ipynb`, `Decision_tree_and_random_forest (1).ipynb`, and `linear_and_KNN_Model_training (1).ipynb` files.

### 1. Data Loading and Preparation

First, load your dataset. This section assumes you have your features (`X`) and target variable (`y`) ready for model training. Replace this with the data loading and preprocessing steps from your notebooks.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.svm import SVR # Support Vector Regressor, assuming regression task due to 'ridge'
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

# --- Placeholder for Data Loading ---
# IMPORTANT: Replace this with your actual data loading and preprocessing code.
# Example: Assuming you have a CSV file or can generate dummy data.

# Generate some dummy data for demonstration purposes
np.random.seed(42)
X = pd.DataFrame(np.random.rand(100, 10), columns=[f'feature_{i}' for i in range(10)])
y = pd.Series(np.random.rand(100) * 10 + X['feature_0'] * 5)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (80, 10)
y_train shape: (80,)
X_test shape: (20, 10)
y_test shape: (20,)


### 2. Model Definitions

Below, I've defined instances for Ridge Regression and Support Vector Regressor (SVR). Please replace or extend these with the exact model configurations you used in your `ridge_and_SVM.ipynb`, `Decision_tree_and_random_forest (1).ipynb`, and `linear_and_KNN_Model_training (1).ipynb` files. Make sure to instantiate your models here.

In [2]:
# --- Model Instances ---
# Replace these with your actual model instances and their initial parameters.

models = {
    'Ridge': Ridge(random_state=42),
    'SVM': SVR(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'K-Nearest Neighbors': KNeighborsRegressor()
}

print("Models initialized:")
for name, model in models.items():
    print(f"- {name}: {model}")

Models initialized:
- Ridge: Ridge(random_state=42)
- SVM: SVR()
- Decision Tree: DecisionTreeRegressor(random_state=42)
- Random Forest: RandomForestRegressor(random_state=42)
- K-Nearest Neighbors: KNeighborsRegressor()


### 3. Pipeline Construction

Pipelines help to streamline the workflow by chaining multiple steps (e.g., preprocessing, model training). Here, we'll create pipelines that include `StandardScaler` for numerical features followed by the respective models. Adapt this if your preprocessing steps are different (e.g., one-hot encoding for categorical features).

In [3]:
pipelines = {}
for name, model in models.items():
    pipelines[name] = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

print("Pipelines created:")
for name, pipeline in pipelines.items():
    print(f"- {name}:\n{pipeline}\n")

Pipelines created:
- Ridge:
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', Ridge(random_state=42))])

- SVM:
Pipeline(steps=[('scaler', StandardScaler()), ('model', SVR())])

- Decision Tree:
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', DecisionTreeRegressor(random_state=42))])

- Random Forest:
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', RandomForestRegressor(random_state=42))])

- K-Nearest Neighbors:
Pipeline(steps=[('scaler', StandardScaler()), ('model', KNeighborsRegressor())])



### 4. Hyperparameter Tuning and Cross-Validation

Now, we'll define hyperparameter grids for tuning two of the models (Ridge and SVM, as requested). You can extend these grids or add more models. We'll use `GridSearchCV` with `KFold` cross-validation to find the best parameters.

In [6]:
param_grids = {
    'Ridge': {
        'model__alpha': [0.1, 1.0, 10.0, 100.0]
    },
    'SVM': {
        'model__C': [0.1, 1, 10, 100],
        'model__gamma': [1, 0.1, 0.01, 0.001],
        'model__kernel': ['rbf']
    },
    'Decision Tree': {
        'model__max_depth': [None, 5, 10, 20]
    },
    'Random Forest': {
        'model__n_estimators': [50, 100, 200],
        'model__max_depth': [None, 5, 10]
    },
    'K-Nearest Neighbors': {
        'model__n_neighbors': [3, 5, 7, 9]
    }
}

# Store results
results = []
best_estimators = {}

# Define cross-validation strategy
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Perform Grid Search for each specified model
for name in models.keys(): # Now iterating through all models
    if name in pipelines and name in param_grids:
        print(f"\n--- Tuning {name} ---")
        grid_search = GridSearchCV(
            estimator=pipelines[name],
            param_grid=param_grids[name],
            cv=kf,
            scoring='neg_mean_squared_error', # Using negative MSE for maximization
            n_jobs=-1, # Use all available cores
            verbose=1
        )
        grid_search.fit(X_train, y_train)

        best_estimators[name] = grid_search.best_estimator_

        y_pred = best_estimators[name].predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            'Model': name,
            'Best Parameters': grid_search.best_params_,
            'Test MSE': mse,
            'Test R2': r2
        })
        print(f"Best parameters for {name}: {grid_search.best_params_}")
        print(f"Test MSE for {name}: {mse:.4f}")
        print(f"Test R2 for {name}: {r2:.4f}")
    else:
        print(f"Skipping {name}: Pipeline or parameter grid not defined.")


--- Tuning Ridge ---
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best parameters for Ridge: {'model__alpha': 100.0}
Test MSE for Ridge: 12.3983
Test R2 for Ridge: 0.1639

--- Tuning SVM ---
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best parameters for SVM: {'model__C': 1, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
Test MSE for SVM: 14.1938
Test R2 for SVM: 0.0428

--- Tuning Decision Tree ---
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best parameters for Decision Tree: {'model__max_depth': 5}
Test MSE for Decision Tree: 14.9649
Test R2 for Decision Tree: -0.0092

--- Tuning Random Forest ---
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best parameters for Random Forest: {'model__max_depth': 5, 'model__n_estimators': 200}
Test MSE for Random Forest: 10.4561
Test R2 for Random Forest: 0.2949

--- Tuning K-Nearest Neighbors ---
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best parameters for K-Nearest Ne

### 5. Main Comparison Table

Finally, here's the comparison table summarizing the performance of the tuned models on the test set. You can expand this to include more models and metrics as you tune them.

In [8]:
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(by='Test R2', ascending=False)

# Add R2 as a percentage
comparison_df['Test R2 (%)'] = (comparison_df['Test R2'] * 100).round(2)

display(comparison_df)

print("\nSummary of Best Estimators:")
for name, estimator in best_estimators.items():
    print(f"- {name}: {estimator}")

,Model,Best Parameters,Test MSE,Test R2,Test R2 (%)
3,Random Forest,"{'model__max_depth': 5, 'model__n_estimators':...",10.456078,0.294890,29.49
4,K-Nearest Neighbors,{'model__n_neighbors': 9},12.187288,0.178146,17.81
0,Ridge,{'model__alpha': 100.0},12.398282,0.163917,16.39
1,SVM,"{'model__C': 1, 'model__gamma': 0.01, 'model__...",14.193831,0.042834,4.28
2,Decision Tree,{'model__max_depth': 5},14.964855,-0.009161,-0.92



Summary of Best Estimators:
- Ridge: Pipeline(steps=[('scaler', StandardScaler()),
                ('model', Ridge(alpha=100.0, random_state=42))])
- SVM: Pipeline(steps=[('scaler', StandardScaler()), ('model', SVR(C=1, gamma=0.01))])
- Decision Tree: Pipeline(steps=[('scaler', StandardScaler()),
                ('model', DecisionTreeRegressor(max_depth=5, random_state=42))])
- Random Forest: Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 RandomForestRegressor(max_depth=5, n_estimators=200,
                                       random_state=42))])
- K-Nearest Neighbors: Pipeline(steps=[('scaler', StandardScaler()),
                ('model', KNeighborsRegressor(n_neighbors=9))])
